<a href="https://colab.research.google.com/github/Danny3636/Generative-AI-Tasks/blob/main/Required_Task_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

From this code and task, I've learned how to generate synthetic fraud transaction data using CTGAN and evaluate its fidelity, utility, and privacy. I've seen that while the synthetic data can mimic real distributions, there's a trade-off in predictive utility, and privacy checks are crucial to detect potential memorization.

# Task 13: Synthetic Fraud Transaction Data Generation with CTGAN

In this notebook we:
1. Load `fraud_transactions.csv` and prepare it for synthetic data generation.
2. Train a **CTGAN** model to generate **5,000 synthetic fraud‑transaction records**.
3. Evaluate synthetic data quality along three pillars:
   - **Fidelity** – KS tests and correlation‑structure comparison.
   - **Utility** – Train‑on‑Synthetic, Test‑on‑Real (TSTR) classification performance.
   - **Privacy** – Nearest‑neighbour memorisation heuristic.

## 1. Install Dependencies

In [1]:
!pip install ctgan sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.1 MB/s eta 0:00:00


## 2. Imports

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score

from scipy.stats import ks_2samp
from sklearn.neighbors import NearestNeighbors

## 3. Load and Inspect the Data

The dataset contains credit‑card transactions with an `is_fraud` label.

**Columns:** `trans_date_trans_time`, `merchant`, `category`, `amt`, `gender`, `state`, `job`, `is_fraud`.

We will drop high‑cardinality text columns (`merchant` – 692 unique values, `job` – 472 unique values) that would slow CTGAN training without adding modelling value. From `trans_date_trans_time` we extract the **hour of day** as a useful numeric feature (fraud patterns often vary by hour).

In [3]:
# ------------------------------------------------------------------
# Upload fraud_transactions.csv to Colab first, or adjust the path.
# ------------------------------------------------------------------
df = pd.read_csv("fraud_transactions.csv")
print(f"Shape: {df.shape}")
df.head()

Shape: (6486, 8)


,trans_date_trans_time,merchant,category,amt,gender,state,job,is_fraud
0,2/27/19 21:32,"fraud_Langosh, Wintheiser and Hyatt",food_dining,83.64,F,TX,"Physicist, medical",0
1,2/13/19 19:41,fraud_Dibbert and Sons,entertainment,79.13,M,PA,Secretary/administrator,0
2,1/11/19 20:03,"fraud_McDermott, Osinski and Morar",home,12.02,F,CA,"Buyer, industrial",0
3,1/20/19 9:08,fraud_Bauch-Raynor,grocery_pos,84.41,M,TN,Clothing/textile technologist,0
4,1/4/19 17:04,"fraud_Reichert, Huels and Hoppe",shopping_net,2.81,F,ME,Financial trader,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6486 entries, 0 to 6485
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   trans_date_trans_time  6486 non-null   object 
 1   merchant               6486 non-null   object 
 2   category               6486 non-null   object 
 3   amt                    6486 non-null   float64
 4   gender                 6486 non-null   object 
 5   state                  6486 non-null   object 
 6   job                    6486 non-null   object 
 7   is_fraud               6486 non-null   int64  
dtypes: float64(1), int64(1), object(6)
memory usage: 405.5+ KB


In [5]:
df["is_fraud"].value_counts(normalize=True)

,proportion
is_fraud,
0,0.925069
1,0.074931


## 4. Feature Engineering & Column Selection

In [6]:
# Extract hour-of-day from the transaction timestamp
df["trans_hour"] = pd.to_datetime(df["trans_date_trans_time"]).dt.hour

# Define column groups
target_col = "is_fraud"
cat_cols   = ["category", "gender", "state"]
num_cols   = ["amt", "trans_hour"]

# Keep only selected columns and drop any NA rows
df = df[cat_cols + num_cols + [target_col]].dropna().reset_index(drop=True)

print(f"Working dataframe shape: {df.shape}")
df.head()

/tmp/ipykernel_5691/1993358421.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["trans_hour"] = pd.to_datetime(df["trans_date_trans_time"]).dt.hour


Working dataframe shape: (6486, 6)


,category,gender,state,amt,trans_hour,is_fraud
0,food_dining,F,TX,83.64,21,0
1,entertainment,M,PA,79.13,19,0
2,home,F,CA,12.02,20,0
3,grocery_pos,M,TN,84.41,9,0
4,shopping_net,F,ME,2.81,17,0


## 5. Train / Test Split on Real Data

We hold out 20 % of the real data as a **test set** that will be used to evaluate both real‑trained and synthetic‑trained classifiers.

In [7]:
real_train, real_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df[target_col]
)

print(f"Real train: {real_train.shape}, Real test: {real_test.shape}")

Real train: (5188, 6), Real test: (1298, 6)


## 6. Train CTGAN on the Real Data

We feed CTGAN all selected columns **including the target** so that the generator learns the joint distribution of features and the fraud label.

In [8]:
from ctgan import CTGAN

ctgan_data = df[cat_cols + num_cols + [target_col]].copy()

discrete_cols = cat_cols + [target_col]  # tell CTGAN which columns are categorical

ctgan = CTGAN(
    epochs=300,
    batch_size=500,
    verbose=True
)

ctgan.fit(ctgan_data, discrete_columns=discrete_cols)

Gen. (-00.71) | Discrim. (-00.04): 100%|██████████| 300/300 [06:05<00:00,  1.22s/it]


## 7. Generate 5,000 Synthetic Records

In [9]:
n_synth = 5_000
synthetic = ctgan.sample(n_synth)

print(f"Synthetic shape: {synthetic.shape}")
synthetic.head()

Synthetic shape: (5000, 6)


,category,gender,state,amt,trans_hour,is_fraud
0,misc_pos,F,MI,6.935697,14,0
1,shopping_pos,M,TN,11.524767,3,0
2,home,M,NC,46.693770,17,0
3,grocery_pos,M,OH,61.505296,7,0
4,misc_pos,F,CA,42.104336,13,0


In [10]:
synthetic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   category    5000 non-null   object 
 1   gender      5000 non-null   object 
 2   state       5000 non-null   object 
 3   amt         5000 non-null   float64
 4   trans_hour  5000 non-null   int32  
 5   is_fraud    5000 non-null   int64  
dtypes: float64(1), int32(1), int64(1), object(3)
memory usage: 215.0+ KB


In [11]:
print("Real outcome distribution:")
print(df[target_col].value_counts(normalize=True))

print("\nSynthetic outcome distribution:")
print(synthetic[target_col].value_counts(normalize=True))

Real outcome distribution:
is_fraud
0    0.925069
1    0.074931
Name: proportion, dtype: float64

Synthetic outcome distribution:
is_fraud
0    0.8322
1    0.1678
Name: proportion, dtype: float64


---
# Evaluation: Is the Synthetic Data Trustworthy?

We evaluate along three pillars:
1. **Fidelity** – statistical similarity to the real data.
2. **Utility** – downstream ML performance (TSTR).
3. **Privacy** – memorisation risk.

## 8. Fidelity Evaluation

### 8.1 Univariate Fidelity – Kolmogorov‑Smirnov Test

For each numeric column we run a two‑sample KS test (real vs synthetic).  
A KS statistic close to **0** means the two distributions are very similar.

In [12]:
real = ctgan_data
syn  = synthetic

ks_results = {}
for col in num_cols:
    r = real[col].sample(min(5000, len(real)), random_state=42)
    s = syn[col].sample(min(5000, len(syn)), random_state=42)
    stat, pval = ks_2samp(r, s)
    ks_results[col] = {"ks_stat": round(stat, 4), "p_value": round(pval, 6)}

ks_df = pd.DataFrame(ks_results).T.sort_values("ks_stat")
ks_df

,ks_stat,p_value
amt,0.0904,0.0
trans_hour,0.0986,0.0


**Interpretation:** Lower KS statistics indicate the synthetic distribution closely matches the real one for that feature. A high p‑value (> 0.05) means we cannot reject the null hypothesis that the two samples come from the same distribution.

### 8.2 Correlation Structure

We compare pairwise Pearson correlations between numeric features in the real and synthetic datasets. Small absolute differences indicate the synthetic data preserves inter‑feature relationships.

In [13]:
real_corr = real[num_cols].corr()
syn_corr  = syn[num_cols].corr()

corr_diff = (real_corr - syn_corr).abs()

print("Real correlation matrix:")
print(real_corr.round(4))
print("\nSynthetic correlation matrix:")
print(syn_corr.round(4))
print("\nAbsolute difference:")
print(corr_diff.round(4))

Real correlation matrix:
               amt  trans_hour
amt         1.0000      0.0714
trans_hour  0.0714      1.0000

Synthetic correlation matrix:
               amt  trans_hour
amt         1.0000      0.3074
trans_hour  0.3074      1.0000

Absolute difference:
              amt  trans_hour
amt         0.000       0.236
trans_hour  0.236       0.000


In [14]:
# Average absolute correlation difference per feature
corr_diff.mean().sort_values(ascending=False)

,0
amt,0.118005
trans_hour,0.118005


### 8.3 Categorical Fidelity – Distribution Comparison

For each categorical column we compare the real and synthetic frequency distributions.

In [15]:
for col in cat_cols:
    print(f"\n--- {col} ---")
    compare = pd.DataFrame({
        "real_pct":      real[col].value_counts(normalize=True),
        "synthetic_pct": syn[col].value_counts(normalize=True)
    }).fillna(0).sort_values("real_pct", ascending=False)
    compare["abs_diff"] = (compare["real_pct"] - compare["synthetic_pct"]).abs()
    print(compare.head(15).round(4))


--- category ---
                real_pct  synthetic_pct  abs_diff
category                                         
gas_transport     0.1065         0.0966    0.0099
grocery_pos       0.1039         0.1124    0.0085
home              0.0928         0.0606    0.0322
shopping_pos      0.0893         0.0904    0.0011
kids_pets         0.0876         0.0804    0.0072
shopping_net      0.0803         0.0636    0.0167
entertainment     0.0722         0.0888    0.0166
food_dining       0.0668         0.0942    0.0274
misc_pos          0.0628         0.0762    0.0134
health_fitness    0.0628         0.0576    0.0052
personal_care     0.0603         0.0424    0.0179
misc_net          0.0506         0.0598    0.0092
grocery_net       0.0330         0.0372    0.0042
travel            0.0313         0.0398    0.0085

--- gender ---
        real_pct  synthetic_pct  abs_diff
gender                                   
F         0.5535          0.533    0.0205
M         0.4465          0.467    0.020

---
## 9. Utility Evaluation – Train on Synthetic, Test on Real (TSTR)

We train a Random Forest classifier to predict `is_fraud`:

| Experiment | Training data | Test data |
|---|---|---|
| **Baseline (TRTR)** | Real train | Real test |
| **TSTR** | Synthetic (5,000 rows) | Real test |

Metrics: **ROC‑AUC** and **F1‑score**.

### 9.1 Preprocessing Pipeline (fit on real train only)

In [16]:
# Fit encoders / scaler on REAL TRAINING data only
ohe    = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
scaler = StandardScaler()

ohe.fit(real_train[cat_cols])
scaler.fit(real_train[num_cols])

def preprocess_for_model(df_subset):
    """Transform a dataframe into model-ready (X, y)."""
    X_cat = ohe.transform(df_subset[cat_cols])
    X_num = scaler.transform(df_subset[num_cols])
    X_all = np.hstack([X_cat, X_num])
    y_out = df_subset[target_col].astype(int)
    return X_all, y_out

### 9.2 Baseline – Train on Real, Test on Real

In [17]:
X_real_train, y_real_train = preprocess_for_model(real_train)
X_real_test,  y_real_test  = preprocess_for_model(real_test)

rf_real = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_real.fit(X_real_train, y_real_train)

y_proba_real = rf_real.predict_proba(X_real_test)[:, 1]
y_pred_real  = (y_proba_real >= 0.5).astype(int)

auc_real = roc_auc_score(y_real_test, y_proba_real)
f1_real  = f1_score(y_real_test, y_pred_real)

print(f"Baseline  –  AUC: {auc_real:.4f}   F1: {f1_real:.4f}")

Baseline  –  AUC: 0.9921   F1: 0.8663


### 9.3 TSTR – Train on Synthetic, Test on Real

In [18]:
X_syn_train, y_syn_train = preprocess_for_model(synthetic)

rf_syn = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_syn.fit(X_syn_train, y_syn_train)

y_proba_syn = rf_syn.predict_proba(X_real_test)[:, 1]
y_pred_syn  = (y_proba_syn >= 0.5).astype(int)

auc_syn = roc_auc_score(y_real_test, y_proba_syn)
f1_syn  = f1_score(y_real_test, y_pred_syn)

print(f"TSTR      –  AUC: {auc_syn:.4f}   F1: {f1_syn:.4f}")

TSTR      –  AUC: 0.8939   F1: 0.6243


### 9.4 Utility Comparison Table

In [19]:
utility_df = pd.DataFrame(
    {
        "AUC": [auc_real, auc_syn],
        "F1":  [f1_real,  f1_syn]
    },
    index=["Train REAL, Test REAL (Baseline)",
           "Train SYNTHETIC, Test REAL (TSTR)"]
)

utility_df

,AUC,F1
"Train REAL, Test REAL (Baseline)",0.992051,0.866310
"Train SYNTHETIC, Test REAL (TSTR)",0.893873,0.624339


**Interpretation:** The closer the TSTR row is to the baseline, the higher the utility of the synthetic data. A large gap indicates the synthetic data does not fully capture the patterns needed to predict fraud.

---
## 10. Privacy Evaluation – Nearest‑Neighbour Memorisation Check

For each synthetic record we find the distance to its **nearest real neighbour** (numeric features only, Euclidean distance). Very small distances could indicate that CTGAN has memorised real rows.

In [20]:
real_num = real[num_cols].sample(min(5000, len(real)), random_state=42)
syn_num  = syn[num_cols].sample(min(5000, len(syn)),  random_state=42)

nn = NearestNeighbors(n_neighbors=1)
nn.fit(real_num)

distances, _ = nn.kneighbors(syn_num)
distances = distances.flatten()

print("Nearest-neighbour distance statistics (synthetic → real):")
pd.Series(distances).describe()

Nearest-neighbour distance statistics (synthetic → real):


,0
count,5000.000000
mean,0.997351
std,2.215845
min,0.000042
25%,0.110316
50%,0.353981
75%,1.009594
max,69.781086


**Interpretation:**
- If the **minimum** distance is very close to zero and many records cluster at tiny distances, CTGAN may be memorising real data points (privacy risk).
- Larger typical distances indicate the synthetic records are novel rather than near‑copies of real transactions.

For production‑grade assurance you would supplement this with membership‑inference attacks or differential privacy variants of CTGAN.

---
## 11. Summary

In this notebook we:

1. Loaded `fraud_transactions.csv` and engineered a compact feature set (category, gender, state, amount, transaction hour, is_fraud).
2. Trained **CTGAN** (300 epochs) on the real data and generated **5,000 synthetic records**.
3. Evaluated the synthetic dataset across three quality pillars:
   - **Fidelity:** KS tests on numeric features, correlation matrix comparison, and categorical distribution comparison.
   - **Utility:** TSTR experiment comparing a Random Forest trained on synthetic vs real data (AUC & F1).
   - **Privacy:** Nearest‑neighbour distance heuristic to detect potential memorisation.